# 03. Mart models
**Case study:** Partisanship & economic perception · **Layer:** marts (intermediate tables → chart- and table-ready facts)

## What a mart does (and does not do)
The intermediate layer made every **coding decision** once (party groups, better/worse binaries, in-party flags, the gap-free
month spine, event clocks). Marts only **aggregate, reshape and compare** those tables so that every number in the story and the
slides can be read straight off one table:
* **No recoding here.** If a mart needs a variable that does not exist upstream, it goes into an intermediate model, not here.
* **One mart per question a reader asks** (one chart, one table, one estimand). Every headline number traces to exactly one mart.
* **Main spec and robustness specs side by side**, as rows or columns, so the robustness battery is a filter, not a re-run.
* **Uncertainty travels with the estimate.** Where SQL can do it honestly (weighted proportions), approximate standard errors
  are columns. Where it cannot (bootstraps, regressions), notebook `04_models.py` writes companion tables.

## Models built here
| # | Mart | Built from | Grain | Answers | RQ |
|---|---|---|---|---|---|
| 1 | `fct_party_ratings` | `int_respondents`, `int_pew_polls`, `int_econ_windows` | source × wave × party definition × party | "What share of each party said the economy got better, every wave?" | RQ1, RQ2 |
| 2 | `fct_wave_contrasts` | `fct_party_ratings`, `int_pew_polls`, `ref_events`, `ref_midterms` | source × wave pair × party definition × measure | "How much did each side move across each transition, net of independents?" (**RQ1 primary descriptive estimand**) | RQ1 |
| 3 | `fct_sentiment_monthly` | `int_sentiment_monthly`, `ref_events`, `ref_midterms` | month × series | Lede chart: monthly party lines with elections and inaugurations marked | RQ1 |
| 4 | `fct_event_study` | `int_event_panel`, `int_sentiment_monthly` | event × month × series | Event-study chart: change from the Jul–Sep baseline, with a noise band | RQ1 |
| 5 | `fct_swing_timing` | `fct_event_study` | event × series | "How much of the swing was done by Election Day / Inauguration Day?" (**co-primary timing estimand**, A8) | RQ1 |
| 6 | `fct_fundamentals_at_transitions` | `int_event_panel` | presidential event × measure | "Did the economy move when opinion flipped?" (fundamentals next to sentiment) | RQ1 |
| 7 | `fct_midterm_placebo` | `fct_event_study`, `fct_wave_contrasts` | source × event × checkpoint | "Does opinion follow Congress too?" (**gap-based** placebo, per intermediate finding 1) | RQ1 validity |
| 8 | `fct_pocketbook_gap` | `int_respondents` (ANES) | survey year × outcome spec × party definition | "Is your wallet less partisan than 'the economy'?" (**RQ3 primary estimand**) | RQ3 |
| 9 | `fct_expected_vs_actual_inflation` | `int_sentiment_monthly`, `int_econ_monthly`, `stg_umich_index` | month | Expected vs actual inflation, CPI components, demographic subgroups | RQ2 |
| 10 | `fct_expected_vs_actual_inflation_by_year` | `fct_expected_vs_actual_inflation` | year (+ all-months row) | The yearly numbers quoted in the plan (−3.0 in 2022, +2.2 in 2025) | RQ2 |
| 11 | `fct_rq2_annual_panel` | `fct_party_ratings`, `int_econ_windows` | CES year × window length | Independents' ratings next to the fundamentals they had just lived through | RQ2 |
| 12 | `fct_say_vs_do_monthly` | `int_sentiment_monthly`, `int_econ_monthly`, `ref_events`, `ref_midterms` | month | Sentiment next to real spending and saving | RQ4 |

**Notebook `04_models.py` (Python)** reads these marts and `int_respondents` and writes: `fct_rq1_model_estimates`,
`fct_rq1_year_effects`, `fct_swing_timing_bootstrap`, `fct_pocketbook_gap_bootstrap`, `fct_rq2_correlations`,
`fct_say_vs_do_correlations`.

## Decisions made in this notebook (each is explained where it happens)
1. **Both "not sure" specs are always reported.** `int_respondents` set them up; marts carry both, labeled. The main spec
   excludes "not sure"; the `_ns_as_same` columns are the plan's A7 robustness check (and what the preliminary numbers used).
2. **Both party definitions are always reported** (leaners with parties = main; leaners as independents = robustness).
3. **Symmetric framing is built into the columns.** Contrasts are expressed as *winning party* vs *losing party* (whichever party
   that was), plus the raw Republican/Democrat numbers. This is the plan's brand-safety rule turned into a schema.
4. **"Net of independents"** (RQ1's estimand) = a party's change minus independents' change over the same two waves, i.e. a
   difference in differences with independents as the benchmark (assumption A2).
5. **Approximate standard errors** for weighted shares use the Kish effective sample size, `n_eff = (Σw)² / Σw²`, and
   `SE = √(p(1 − p) / n_eff)`. Neither CES nor this ANES file carries design variables (staging gotcha 1), so these are labeled
   `_approx` and treated as a floor on the true uncertainty. Bootstraps in notebook 04 are the reported intervals.
6. **Monthly noise (UMich has no SEs, assumption A9)** is estimated once per series, robustly: the median absolute deviation
   (MAD) of month-to-month changes over the whole series, scaled to a standard deviation (× 1.4826) and divided by √2 to get the
   noise in a single month's reading. MAD ignores the big transition and COVID moves without anyone choosing which months to drop,
   which keeps the choice pre-specified. It treats all month-to-month movement in calm periods as noise, so it is conservative.
7. **Event checkpoints are fixed in months from the election**, so presidential elections and midterms line up:
   0 = election month, +2 = takeover month (January: inauguration, or the new Congress sworn in), +3 = three months after,
   +8 = six months after takeover. The swing denominator is always the change from the Jul–Sep baseline to +8.
8. **The midterm placebo is gap-based** (intermediate finding 1): in 2022 every group rose 8–11 points together within three
   months, so party *levels* moved; the estimand is the change in the **winning-minus-losing gap**, which barely moved.

## Conventions (same as notebooks 01 and 02)
* Run notebooks 01 and 02 first, then this notebook top to bottom. Every cell is `CREATE OR REPLACE`, so re-running is safe.
* Tables land in `workspace.public_opinion_economy_case_study` with an `fct_` prefix.
* Percentages are stored as **percent** (52.5 = 52.5%) rounded to 2 decimals; QA cells round to 1 to match the plan.
* Every division is wrapped in `try_divide` or guarded with `NULLIF(…, 0)`: serverless runs in ANSI mode, where dividing by zero
  raises an error instead of returning NULL (e.g. CES has no own-finances item, so its denominator is 0).
* **dbt mapping:** each `SELECT` is the body of `dbt/models/marts/<name>.sql`; swap table names for `{{ ref('...') }}`. Marts that
  read other marts (`fct_wave_contrasts`, `fct_swing_timing`, `fct_midterm_placebo`, …) do so on purpose: one aggregation,
  several shapes, so two charts can never disagree. The QA cells at the end become `schema.yml` tests and singular tests.

# 0. Setup

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA public_opinion_economy_case_study;

-- Same as notebooks 01-02: timestamps become dates in UTC, so nothing depends on the cluster's time zone.
SET TIME ZONE 'UTC';

## 1. `fct_party_ratings` — every survey wave, by party
**Grain:** one row per source × wave × party definition × party group.
* **CES / ANES:** one wave per survey year. Party groups: Democrat, Republican, Independent, All respondents.
  Party definitions: *Leaners with parties* (main) and *Leaners as independents* (robustness).
* **Pew:** one wave per poll (28). Party groups: Democrat, Republican, All respondents. Pew groups leaners with parties only.

This is the table behind the CES and ANES descriptive charts, the Pew cross-check line, and (via `fct_wave_contrasts`) the
RQ1 transition contrasts.

### Never compare levels across sources (assumption A4)
CES and ANES ask whether the economy got **better or worse over the past year**; Pew asks whether it **is excellent or good now**.
`pct_positive` puts each source's headline positive share in one column so the three lines can share a chart, and
`positive_measure` says what it is. Compare **changes and gaps** across sources, never levels.

### Columns
| Column group | Columns | Notes |
|---|---|---|
| Main spec ("not sure" excluded) | `pct_better`, `pct_worse`, `net_rating` | `net_rating` = % better − % worse |
| Robustness | `pct_better_ns_as_same`, `pct_worse_ns_as_same`, `mean_score5` (5-point mean, 5 = much better), `pct_better_unweighted` | |
| Uncertainty | `se_pct_better_approx`, `se_pct_worse_approx`, `se_net_rating_approx`, `n_eff_main` | Kish approximation (decision 5) |
| Own finances (ANES only) | `personal_*` | NULL for CES and Pew |
| Pew only | `pct_excellent_good`, `pew_*` flags | toplines, no SEs in source |

Weighted share of a 0/1 column = `Σ(x·w) / Σ(w over non-NULL x)`. NULLs (e.g. "not sure" in the main spec) drop out of both
numerator and denominator, which is exactly what "excluded" means.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_party_ratings
COMMENT 'Economic ratings by party for every survey wave: CES and ANES by year (main and robustness specs, approximate SEs, ANES own finances) and Pew by poll. One row per source x wave x party definition x party group.'
AS
WITH
-- ─────────────────────────────────────────────────────────────────────────────
-- Fieldwork date per CES/ANES wave: the median respondent start date (CES) or the assumed 1 October (ANES),
-- taken from int_econ_windows so every model uses the same wave date.
-- ─────────────────────────────────────────────────────────────────────────────
waves AS (
  SELECT source, survey_year, fieldwork_median_date, fieldwork_date_is_assumed
  FROM int_econ_windows
  WHERE is_primary_window                       -- the window rows repeat the fieldwork date; keep one per wave
),

-- ─────────────────────────────────────────────────────────────────────────────
-- Respondent rows, one copy per (party definition, party group) they belong to.
-- stack() emits four rows per respondent:
--   1. main definition, own party        2. main definition, "All respondents"
--   3. strict definition, own party      4. strict definition, "All respondents"
-- so the main spec, the leaner robustness check and the all-respondents benchmark share one code path.
-- ─────────────────────────────────────────────────────────────────────────────
respondents_by_group AS (
  SELECT
    r.source,
    r.survey_year,
    r.weight,
    r.white_house_party,
    g.party_definition,
    g.party_group,
    g.partisan_role,
    -- national economy (both surveys)
    r.econ_better, r.econ_worse, r.econ_net, r.econ_better_ns_as_same, r.econ_worse_ns_as_same, r.econ_score5,
    -- own finances (ANES only; NULL in CES)
    r.personal_better, r.personal_worse, r.personal_net,
    r.personal_better_ns_as_same, r.personal_worse_ns_as_same, r.personal_score5
  FROM int_respondents AS r
  LATERAL VIEW stack(4,
      'Leaners with parties',    r.party3,          r.partisan_role,
      'Leaners with parties',    'All respondents', 'All respondents',
      'Leaners as independents', r.party3_strict,   r.partisan_role_strict,
      'Leaners as independents', 'All respondents', 'All respondents'
  ) g AS party_definition, party_group, partisan_role
  -- "Not sure" / missing party only appear inside "All respondents"; they are not a party group of their own.
  WHERE g.party_group IN ('Democrat', 'Republican', 'Independent', 'All respondents')
),

-- ─────────────────────────────────────────────────────────────────────────────
-- Weighted aggregation for CES and ANES. Each share's denominator is the weight of respondents with a non-NULL value
-- for THAT column, so main-spec and robustness shares each use their own correct base.
-- ─────────────────────────────────────────────────────────────────────────────
survey_aggregates AS (
  SELECT
    source,
    survey_year,
    party_definition,
    party_group,
    MAX(partisan_role)                                                                         AS partisan_role,
    MAX(white_house_party)                                                                     AS white_house_party,  -- constant within a wave (QA 13b)

    -- National economy: main spec ("not sure" excluded)
    100 * SUM(econ_better * weight) / NULLIF(SUM(IF(econ_better IS NOT NULL, weight, 0)), 0)  AS pct_better,
    100 * SUM(econ_worse  * weight) / NULLIF(SUM(IF(econ_worse  IS NOT NULL, weight, 0)), 0)  AS pct_worse,
    100 * SUM(econ_net    * weight) / NULLIF(SUM(IF(econ_net    IS NOT NULL, weight, 0)), 0)  AS net_rating,

    -- National economy: robustness
    100 * SUM(econ_better_ns_as_same * weight)
        / NULLIF(SUM(IF(econ_better_ns_as_same IS NOT NULL, weight, 0)), 0)                   AS pct_better_ns_as_same,
    100 * SUM(econ_worse_ns_as_same * weight)
        / NULLIF(SUM(IF(econ_worse_ns_as_same IS NOT NULL, weight, 0)), 0)                    AS pct_worse_ns_as_same,
    SUM(econ_score5 * weight) / NULLIF(SUM(IF(econ_score5 IS NOT NULL, weight, 0)), 0)        AS mean_score5,
    100 * AVG(econ_better)                                                                     AS pct_better_unweighted,  -- AVG skips NULLs

    -- Sample sizes and Kish effective n for the main-spec base
    COUNT(*)                                                                                   AS n_respondents,
    COUNT(econ_better)                                                                         AS n_valid_main,
    POWER(SUM(IF(econ_better IS NOT NULL, weight, 0)), 2)
        / NULLIF(SUM(IF(econ_better IS NOT NULL, weight * weight, 0)), 0)                     AS n_eff_main,

    -- Own finances (ANES only): same formulas
    100 * SUM(personal_better * weight) / NULLIF(SUM(IF(personal_better IS NOT NULL, weight, 0)), 0) AS personal_pct_better,
    100 * SUM(personal_worse  * weight) / NULLIF(SUM(IF(personal_worse  IS NOT NULL, weight, 0)), 0) AS personal_pct_worse,
    100 * SUM(personal_net    * weight) / NULLIF(SUM(IF(personal_net    IS NOT NULL, weight, 0)), 0) AS personal_net_rating,
    100 * SUM(personal_better_ns_as_same * weight)
        / NULLIF(SUM(IF(personal_better_ns_as_same IS NOT NULL, weight, 0)), 0)               AS personal_pct_better_ns_as_same,
    100 * SUM(personal_worse_ns_as_same * weight)
        / NULLIF(SUM(IF(personal_worse_ns_as_same IS NOT NULL, weight, 0)), 0)                AS personal_pct_worse_ns_as_same,
    SUM(personal_score5 * weight) / NULLIF(SUM(IF(personal_score5 IS NOT NULL, weight, 0)), 0) AS personal_mean_score5,
    COUNT(personal_better)                                                                     AS personal_n_valid_main
  FROM respondents_by_group
  GROUP BY source, survey_year, party_definition, party_group
),

survey_rows AS (
  SELECT
    a.source,
    concat(a.source, '-', CAST(a.survey_year AS STRING))                AS wave_key,
    a.survey_year,
    w.fieldwork_median_date                                              AS wave_date,
    w.fieldwork_date_is_assumed                                          AS wave_date_is_assumed,   -- TRUE for ANES (1 Oct assumed)
    a.white_house_party,
    a.party_definition,
    a.party_group,
    a.partisan_role,
    'Economy got better over the past year (% of those answering)'       AS positive_measure,
    ROUND(a.pct_better, 2)                                               AS pct_positive,
    ROUND(a.pct_better, 2)                                               AS pct_better,
    ROUND(a.pct_worse, 2)                                                AS pct_worse,
    ROUND(a.net_rating, 2)                                               AS net_rating,
    ROUND(a.pct_better_ns_as_same, 2)                                    AS pct_better_ns_as_same,
    ROUND(a.pct_worse_ns_as_same, 2)                                     AS pct_worse_ns_as_same,
    ROUND(a.mean_score5, 3)                                              AS mean_score5,
    ROUND(a.pct_better_unweighted, 2)                                    AS pct_better_unweighted,
    -- Approximate SEs (decision 5). For the net rating: Var(better − worse) = (pb + pw − (pb − pw)²) / n_eff.
    ROUND(100 * sqrt((a.pct_better / 100) * (1 - a.pct_better / 100) / a.n_eff_main), 2) AS se_pct_better_approx,
    ROUND(100 * sqrt((a.pct_worse  / 100) * (1 - a.pct_worse  / 100) / a.n_eff_main), 2) AS se_pct_worse_approx,
    ROUND(100 * sqrt(((a.pct_better + a.pct_worse) / 100 - POWER(a.net_rating / 100, 2)) / a.n_eff_main), 2)
                                                                         AS se_net_rating_approx,
    a.n_respondents,
    a.n_valid_main,
    ROUND(a.n_eff_main, 0)                                               AS n_eff_main,
    ROUND(a.personal_pct_better, 2)                                      AS personal_pct_better,
    ROUND(a.personal_pct_worse, 2)                                       AS personal_pct_worse,
    ROUND(a.personal_net_rating, 2)                                      AS personal_net_rating,
    ROUND(a.personal_pct_better_ns_as_same, 2)                           AS personal_pct_better_ns_as_same,
    ROUND(a.personal_pct_worse_ns_as_same, 2)                            AS personal_pct_worse_ns_as_same,
    ROUND(a.personal_mean_score5, 3)                                     AS personal_mean_score5,
    IF(a.source = 'ANES', a.personal_n_valid_main, NULL)                 AS personal_n_valid_main,
    CAST(NULL AS DOUBLE)                                                 AS pct_excellent_good,
    CAST(NULL AS STRING)                                                 AS pew_transition_label,
    CAST(NULL AS STRING)                                                 AS pew_poll_period,
    CAST(NULL AS BOOLEAN)                                                AS pew_is_dated_on_election_day
  FROM survey_aggregates AS a
  JOIN waves AS w
    ON a.source = w.source AND a.survey_year = w.survey_year
),

-- ─────────────────────────────────────────────────────────────────────────────
-- Pew: already one value per poll x series in int_pew_polls; only relabeled to the shared schema.
-- ─────────────────────────────────────────────────────────────────────────────
pew_rows AS (
  SELECT
    'Pew'                                                                AS source,
    concat('Pew-', CAST(poll_date AS STRING))                            AS wave_key,
    year(poll_date)                                                      AS survey_year,
    poll_date                                                            AS wave_date,
    FALSE                                                                AS wave_date_is_assumed,   -- but fieldwork vs publication is unverified
    white_house_party,
    'Leaners with parties'                                               AS party_definition,       -- Pew's only grouping
    IF(series = 'Total', 'All respondents', series)                      AS party_group,
    partisan_role,
    'Economy is excellent or good now (%)'                               AS positive_measure,
    pct_excellent_good                                                   AS pct_positive,
    CAST(NULL AS DOUBLE) AS pct_better,             CAST(NULL AS DOUBLE) AS pct_worse,
    CAST(NULL AS DOUBLE) AS net_rating,             CAST(NULL AS DOUBLE) AS pct_better_ns_as_same,
    CAST(NULL AS DOUBLE) AS pct_worse_ns_as_same,   CAST(NULL AS DOUBLE) AS mean_score5,
    CAST(NULL AS DOUBLE) AS pct_better_unweighted,
    CAST(NULL AS DOUBLE) AS se_pct_better_approx,   CAST(NULL AS DOUBLE) AS se_pct_worse_approx,
    CAST(NULL AS DOUBLE) AS se_net_rating_approx,
    CAST(NULL AS BIGINT) AS n_respondents,          CAST(NULL AS BIGINT) AS n_valid_main,
    CAST(NULL AS DOUBLE) AS n_eff_main,
    CAST(NULL AS DOUBLE) AS personal_pct_better,    CAST(NULL AS DOUBLE) AS personal_pct_worse,
    CAST(NULL AS DOUBLE) AS personal_net_rating,    CAST(NULL AS DOUBLE) AS personal_pct_better_ns_as_same,
    CAST(NULL AS DOUBLE) AS personal_pct_worse_ns_as_same,
    CAST(NULL AS DOUBLE) AS personal_mean_score5,   CAST(NULL AS BIGINT) AS personal_n_valid_main,
    pct_excellent_good,
    transition_label                                                     AS pew_transition_label,
    poll_period                                                          AS pew_poll_period,
    is_dated_on_election_day                                             AS pew_is_dated_on_election_day
  FROM int_pew_polls
)

SELECT * FROM survey_rows
UNION ALL
SELECT * FROM pew_rows;

## 2. `fct_wave_contrasts` — how far each side moved between two waves
**Grain:** one row per source × wave pair × party definition × measure.

This is the **RQ1 primary descriptive estimand**: *the change in the Republican–Democrat gap across each transition, net of
independents.* Built on `fct_party_ratings`, so the contrast and the levels chart can never disagree.

### Which wave pairs
| Source | Pairs | Why |
|---|---|---|
| CES | every consecutive year, 2016→2017 … 2024→2025 (9 pairs) | transitions, midterms and quiet years all in one table, labeled by `comparison_type`, so quiet years show the normal year-to-year drift |
| ANES | 2016→2020, 2020→2024 | each spans a transition (and a midterm); flagged in `caveats` |
| Pew | per transition, two **pre-specified** windows | (a) *last poll before vs first poll after inauguration*: the plan's Pew design. (b) *pre-election to six months after inauguration*: the same window as the UMich swing denominator (decision 7), so Pew and UMich measure the same thing |

### Classifying each pair (`comparison_type`)
Checked in this order: **Presidential transition** (White House party differs between the two waves) → **Midterm** (a midterm
election fell between the waves) → **Presidential re-election** → **No federal election between waves**.
`winning_party` is the party that took the White House (transition) or the flipped chamber (midterm).

### Columns that encode the estimand
* `rep_change`, `dem_change`, `ind_change`, `all_change`: each group's change, t0 → t1.
* `winning_party_change`, `losing_party_change`: the same, relabeled symmetrically (decision 3).
* `winning_minus_ind_change`, `losing_minus_ind_change`: **net of independents** (decision 4). Positive for the winner and
  negative for the loser is the "perceptual screen" pattern; ≈ 0 for both is what the falsification test looks for.
* `winner_minus_loser_gap_change` = `winning_party_change − losing_party_change` (identical to the change in the gap). Its
  approximate SE (`se_winner_minus_loser_gap_change`) combines four independent shares (two parties × two waves); available for the
  three measures that have an SE in `fct_party_ratings`.
* `measure` covers the robustness battery: main binary, "not sure" as same, 5-point mean, net rating, % worse, unweighted, Pew %.
  `measure_unit` says whether a change is in percentage points or scale points.
* **`quiet_years_mean_abs_gap_change` / `_max_`: the empirical yardstick.** Sampling SEs for CES are 1–2 points, but in the four
  CES pairs with **no** federal election between waves (2017→18, 2019→20, 2021→22, 2023→24) the Rep–Dem gap still moved 12–17
  points. That non-sampling drift (fieldwork season, news cycle) is the honest benchmark for any annual comparison: transitions
  move the gap ~77–95 points, five to six times the quiet-year drift. Found while testing this notebook; see the handoff.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_wave_contrasts
COMMENT 'RQ1 contrasts: change in each party''s rating between two survey waves (CES consecutive years, ANES cycles, Pew pre-specified windows around inaugurations), the gap change, changes net of independents, winning vs losing party framing, approximate SEs.'
AS
WITH
-- ─────────────────────────────────────────────────────────────────────────────
-- 1. Measures to long format (one row per wave x party definition x party group x measure).
--    Every value is cast to DOUBLE because stack() needs one type per output column.
-- ─────────────────────────────────────────────────────────────────────────────
measures_long AS (
  SELECT
    r.source, r.wave_key, r.survey_year, r.wave_date, r.white_house_party, r.party_definition, r.party_group,
    m.measure, m.measure_label, m.measure_unit, m.value, m.se
  FROM fct_party_ratings AS r
  LATERAL VIEW stack(7,
    'pct_better',            'Better (main: not sure excluded)',  'percentage points', CAST(r.pct_better AS DOUBLE),            CAST(r.se_pct_better_approx AS DOUBLE),
    'pct_worse',             'Worse (main: not sure excluded)',   'percentage points', CAST(r.pct_worse AS DOUBLE),             CAST(r.se_pct_worse_approx AS DOUBLE),
    'net_rating',            'Net rating (% better - % worse)',   'percentage points', CAST(r.net_rating AS DOUBLE),            CAST(r.se_net_rating_approx AS DOUBLE),
    'pct_better_ns_as_same', 'Better (not sure counted as same)', 'percentage points', CAST(r.pct_better_ns_as_same AS DOUBLE), CAST(NULL AS DOUBLE),
    'mean_score5',           '5-point mean (5 = much better)',    'scale points',      CAST(r.mean_score5 AS DOUBLE),           CAST(NULL AS DOUBLE),
    'pct_better_unweighted', 'Better (main, unweighted)',         'percentage points', CAST(r.pct_better_unweighted AS DOUBLE), CAST(NULL AS DOUBLE),
    'pct_excellent_good',    'Excellent or good now (Pew)',       'percentage points', CAST(r.pct_excellent_good AS DOUBLE),    CAST(NULL AS DOUBLE)
  ) m AS measure, measure_label, measure_unit, value, se
  WHERE m.value IS NOT NULL                        -- drops Pew rows for survey measures and vice versa
),

-- 2. Parties side by side: one row per wave x party definition x measure.
wide AS (
  SELECT
    source, wave_key, survey_year, wave_date, party_definition, measure, measure_label, measure_unit,
    MAX(white_house_party)                                         AS white_house_party,
    MAX(IF(party_group = 'Republican',      value, NULL))          AS rep,
    MAX(IF(party_group = 'Democrat',        value, NULL))          AS dem,
    MAX(IF(party_group = 'Independent',     value, NULL))          AS ind,
    MAX(IF(party_group = 'All respondents', value, NULL))          AS all_resp,
    MAX(IF(party_group = 'Republican',      se, NULL))             AS rep_se,
    MAX(IF(party_group = 'Democrat',        se, NULL))             AS dem_se,
    MAX(IF(party_group = 'Independent',     se, NULL))             AS ind_se
  FROM measures_long
  GROUP BY source, wave_key, survey_year, wave_date, party_definition, measure, measure_label, measure_unit
),

-- ─────────────────────────────────────────────────────────────────────────────
-- 3a. CES and ANES: each wave paired with the previous wave of the same survey.
-- ─────────────────────────────────────────────────────────────────────────────
pairs_consecutive AS (
  SELECT
    source, party_definition, measure, measure_label, measure_unit,
    'Consecutive survey waves'                     AS comparison_window,
    LAG(wave_key)          OVER w                  AS wave_key_t0,
    wave_key                                       AS wave_key_t1,
    LAG(wave_date)         OVER w                  AS date_t0,
    wave_date                                      AS date_t1,
    LAG(white_house_party) OVER w                  AS white_house_party_t0,
    white_house_party                              AS white_house_party_t1,
    LAG(rep)      OVER w AS rep_t0,      rep      AS rep_t1,
    LAG(dem)      OVER w AS dem_t0,      dem      AS dem_t1,
    LAG(ind)      OVER w AS ind_t0,      ind      AS ind_t1,
    LAG(all_resp) OVER w AS all_t0,      all_resp AS all_t1,
    LAG(rep_se)   OVER w AS rep_se_t0,   rep_se   AS rep_se_t1,
    LAG(dem_se)   OVER w AS dem_se_t0,   dem_se   AS dem_se_t1,
    LAG(ind_se)   OVER w AS ind_se_t0,   ind_se   AS ind_se_t1
  FROM wide
  WHERE source IN ('CES', 'ANES')
  WINDOW w AS (PARTITION BY source, party_definition, measure ORDER BY wave_date)
),

-- ─────────────────────────────────────────────────────────────────────────────
-- 3b. Pew: two pre-specified windows per party-changing transition (see markdown above).
--     int_pew_polls already assigned each poll to its nearest transition and flagged the bracketing polls.
-- ─────────────────────────────────────────────────────────────────────────────
pew_polls AS (
  SELECT DISTINCT poll_date, transition_label, election_date, inauguration_date, days_from_inauguration,
                  is_last_before_inauguration, is_first_after_inauguration
  FROM int_pew_polls
),

pew_pair_dates AS (
  -- (a) the plan's design: last poll before vs first poll after the inauguration
  SELECT
    transition_label,
    'Last poll before vs first poll after inauguration'          AS comparison_window,
    MAX(IF(is_last_before_inauguration, poll_date, NULL))        AS date_t0,
    MAX(IF(is_first_after_inauguration, poll_date, NULL))        AS date_t1
  FROM pew_polls
  GROUP BY transition_label

  UNION ALL

  -- (b) pre-election to six months after inauguration: same window as the UMich swing denominator.
  --     "Pre-election" = last poll dated on or before Election Day (2016's only candidate is dated Election Day; flagged below).
  SELECT
    transition_label,
    'Pre-election to six months after inauguration',
    MAX(IF(poll_date <= election_date, poll_date, NULL)),
    MAX(IF(days_from_inauguration BETWEEN 0 AND 181, poll_date, NULL))   -- latest poll within ~6 months of the oath
  FROM pew_polls
  GROUP BY transition_label
),

pairs_pew AS (
  SELECT
    'Pew' AS source, w0.party_definition, w0.measure, w0.measure_label, w0.measure_unit,
    p.comparison_window,
    w0.wave_key AS wave_key_t0, w1.wave_key AS wave_key_t1,
    w0.wave_date AS date_t0,    w1.wave_date AS date_t1,
    w0.white_house_party AS white_house_party_t0, w1.white_house_party AS white_house_party_t1,
    w0.rep AS rep_t0, w1.rep AS rep_t1,
    w0.dem AS dem_t0, w1.dem AS dem_t1,
    w0.ind AS ind_t0, w1.ind AS ind_t1,                 -- NULL: Pew file has no independents
    w0.all_resp AS all_t0, w1.all_resp AS all_t1,
    w0.rep_se AS rep_se_t0, w1.rep_se AS rep_se_t1,     -- NULL: no sample sizes in the Pew file (data gap)
    w0.dem_se AS dem_se_t0, w1.dem_se AS dem_se_t1,
    w0.ind_se AS ind_se_t0, w1.ind_se AS ind_se_t1
  FROM pew_pair_dates AS p
  JOIN wide AS w0 ON w0.source = 'Pew' AND w0.wave_date = p.date_t0
  JOIN wide AS w1 ON w1.source = 'Pew' AND w1.wave_date = p.date_t1
                 AND w1.party_definition = w0.party_definition AND w1.measure = w0.measure
),

pairs AS (
  SELECT * FROM pairs_consecutive WHERE wave_key_t0 IS NOT NULL      -- first wave of each survey has nothing to compare with
  UNION ALL
  SELECT * FROM pairs_pew
),

-- ─────────────────────────────────────────────────────────────────────────────
-- 4. Which elections fell between the two waves. Lower bound inclusive so a poll dated Election Day (Pew 2016) counts.
--    A presidential event also counts when only its INAUGURATION falls between the waves: Pew's 2020-21 "last poll before
--    inauguration" (17 Nov 2020) is already after Election Day, but the pair still brackets the change of power.
-- ─────────────────────────────────────────────────────────────────────────────
distinct_pairs AS (
  SELECT DISTINCT source, comparison_window, wave_key_t0, wave_key_t1, date_t0, date_t1 FROM pairs
),

presidential_between AS (
  SELECT d.source, d.comparison_window, d.wave_key_t0, d.wave_key_t1,
         MAX_BY(e.election_year, e.election_date)        AS pres_election_year,
         MAX_BY(e.transition_label, e.election_date)     AS pres_transition_label,
         MAX_BY(e.election_month, e.election_date)       AS pres_election_month
  FROM distinct_pairs AS d
  JOIN ref_events AS e
    ON (e.election_date     >= d.date_t0 AND e.election_date     <  d.date_t1)
    OR (e.inauguration_date >  d.date_t0 AND e.inauguration_date <= d.date_t1)
  GROUP BY d.source, d.comparison_window, d.wave_key_t0, d.wave_key_t1
),

midterm_between AS (
  SELECT d.source, d.comparison_window, d.wave_key_t0, d.wave_key_t1,
         MAX_BY(m.election_year,  m.election_date)       AS mid_election_year,
         MAX_BY(m.winning_party,  m.election_date)       AS mid_winning_party,
         MAX_BY(m.election_month, m.election_date)       AS mid_election_month
  FROM distinct_pairs AS d
  JOIN ref_midterms AS m ON m.election_date >= d.date_t0 AND m.election_date < d.date_t1
  GROUP BY d.source, d.comparison_window, d.wave_key_t0, d.wave_key_t1
),

classified AS (
  SELECT
    p.*,
    pb.pres_election_year, pb.pres_transition_label, pb.pres_election_month,
    mb.mid_election_year, mb.mid_winning_party, mb.mid_election_month,
    CASE
      WHEN p.white_house_party_t0 <> p.white_house_party_t1 THEN 'Presidential transition'
      WHEN mb.mid_election_year IS NOT NULL                 THEN 'Midterm (a chamber of Congress changed hands)'
      WHEN pb.pres_election_year IS NOT NULL                THEN 'Presidential re-election'
      ELSE 'No federal election between waves'
    END                                                     AS comparison_type
  FROM pairs AS p
  LEFT JOIN presidential_between AS pb
    ON  p.source = pb.source AND p.comparison_window = pb.comparison_window
    AND p.wave_key_t0 = pb.wave_key_t0 AND p.wave_key_t1 = pb.wave_key_t1
  LEFT JOIN midterm_between AS mb
    ON  p.source = mb.source AND p.comparison_window = mb.comparison_window
    AND p.wave_key_t0 = mb.wave_key_t0 AND p.wave_key_t1 = mb.wave_key_t1
),

-- 5. Event identity and the symmetric winner / loser labels.
with_winner AS (
  SELECT
    c.*,
    CASE c.comparison_type
      WHEN 'Presidential transition'                       THEN concat('pres-', CAST(c.pres_election_year AS STRING))
      WHEN 'Midterm (a chamber of Congress changed hands)' THEN concat('mid-',  CAST(c.mid_election_year  AS STRING))
      WHEN 'Presidential re-election'                      THEN concat('pres-', CAST(c.pres_election_year AS STRING))
    END                                                    AS event_key,     -- joins to fct_event_study / int_event_panel
    CASE c.comparison_type
      WHEN 'Presidential transition'                       THEN c.pres_transition_label
      WHEN 'Midterm (a chamber of Congress changed hands)' THEN concat(CAST(c.mid_election_year AS STRING), ' midterm')
      WHEN 'Presidential re-election'                      THEN c.pres_transition_label
    END                                                    AS event_label,
    CASE c.comparison_type
      WHEN 'Presidential transition'                       THEN c.pres_election_month
      WHEN 'Midterm (a chamber of Congress changed hands)' THEN c.mid_election_month
      WHEN 'Presidential re-election'                      THEN c.pres_election_month
    END                                                    AS event_election_month,
    CASE c.comparison_type
      WHEN 'Presidential transition'                       THEN c.white_house_party_t1
      WHEN 'Midterm (a chamber of Congress changed hands)' THEN c.mid_winning_party
    END                                                    AS winning_party
  FROM classified AS c
)

SELECT
  -- ── Identity ───────────────────────────────────────────────────────────────
  source,
  comparison_window,
  wave_key_t0,
  wave_key_t1,
  date_t0,
  date_t1,
  comparison_type,
  event_key,
  event_label,
  CASE WHEN event_key LIKE 'pres-%' THEN 'Presidential election'
       WHEN event_key LIKE 'mid-%'  THEN 'Midterm election' END            AS event_type,
  -- Months from the election in between to the second wave (CES odd-year waves land about 12 months later)
  CAST(months_between(trunc(date_t1, 'MM'), event_election_month) AS INT) AS months_from_election_to_t1,
  white_house_party_t0,
  white_house_party_t1,
  winning_party,
  CASE winning_party WHEN 'Republican' THEN 'Democrat' WHEN 'Democrat' THEN 'Republican' END AS losing_party,

  -- ── Flags for the robustness battery and the write-up ─────────────────────
  year(date_t0) IN (2020, 2021) OR year(date_t1) IN (2020, 2021)          AS is_covid_comparison,     -- "excluding 2020-2021"
  comparison_type = 'Presidential transition'
    AND NOT (year(date_t0) IN (2020, 2021) OR year(date_t1) IN (2020, 2021)) AS is_clean_transition,   -- plan: 2016-17 and 2024-25
  concat_ws('; ',
    IF(source = 'ANES', 'ANES waves are four years apart and also span a midterm', NULL),
    IF(source = 'Pew' AND date_t0 = to_date(concat(CAST(year(date_t0) AS STRING), '-11-08')) AND year(date_t0) = 2016,
       'Pew 2016 "before" poll is dated Election Day; fieldwork vs publication date unverified', NULL),
    IF(source = 'Pew' AND comparison_window LIKE 'Last poll%' AND event_key = 'pres-2024',
       'No Pew poll between the 2024 election and inauguration; "before" poll is from September', NULL),
    IF(source = 'CES' AND year(date_t0) % 2 = 0,
       'CES even-year wave fielded during the campaign (Sep-Nov), odd-year wave after it (Nov-Dec)', NULL)
  )                                                                         AS caveats,

  -- ── Measure ────────────────────────────────────────────────────────────────
  party_definition,
  measure,
  measure_label,
  measure_unit,

  -- ── Levels and changes by party ────────────────────────────────────────────
  ROUND(rep_t0, 2) AS rep_t0,  ROUND(rep_t1, 2) AS rep_t1,  ROUND(rep_t1 - rep_t0, 2) AS rep_change,
  ROUND(dem_t0, 2) AS dem_t0,  ROUND(dem_t1, 2) AS dem_t1,  ROUND(dem_t1 - dem_t0, 2) AS dem_change,
  ROUND(ind_t0, 2) AS ind_t0,  ROUND(ind_t1, 2) AS ind_t1,  ROUND(ind_t1 - ind_t0, 2) AS ind_change,
  ROUND(all_t0, 2) AS all_t0,  ROUND(all_t1, 2) AS all_t1,  ROUND(all_t1 - all_t0, 2) AS all_change,

  -- ── The partisan gap ───────────────────────────────────────────────────────
  ROUND(rep_t0 - dem_t0, 2)                                               AS rep_minus_dem_gap_t0,
  ROUND(rep_t1 - dem_t1, 2)                                               AS rep_minus_dem_gap_t1,
  ROUND((rep_t1 - dem_t1) - (rep_t0 - dem_t0), 2)                         AS rep_minus_dem_gap_change,

  -- ── Symmetric framing: winning vs losing party (NULL when no one won anything between waves) ─────
  ROUND(CASE winning_party WHEN 'Republican' THEN rep_t1 - rep_t0 WHEN 'Democrat' THEN dem_t1 - dem_t0 END, 2)
                                                                          AS winning_party_change,
  ROUND(CASE winning_party WHEN 'Republican' THEN dem_t1 - dem_t0 WHEN 'Democrat' THEN rep_t1 - rep_t0 END, 2)
                                                                          AS losing_party_change,
  -- Net of independents (decision 4): a difference in differences with independents as the benchmark
  ROUND(CASE winning_party WHEN 'Republican' THEN (rep_t1 - rep_t0) WHEN 'Democrat' THEN (dem_t1 - dem_t0) END
        - (ind_t1 - ind_t0), 2)                                           AS winning_minus_ind_change,
  ROUND(CASE winning_party WHEN 'Republican' THEN (dem_t1 - dem_t0) WHEN 'Democrat' THEN (rep_t1 - rep_t0) END
        - (ind_t1 - ind_t0), 2)                                           AS losing_minus_ind_change,
  ROUND(CASE winning_party WHEN 'Republican' THEN ((rep_t1 - dem_t1) - (rep_t0 - dem_t0))
                           WHEN 'Democrat'   THEN ((dem_t1 - rep_t1) - (dem_t0 - rep_t0)) END, 2)
                                                                          AS winner_minus_loser_gap_change,

  -- ── Approximate uncertainty (independent samples across waves and parties) ──
  ROUND(sqrt(rep_se_t0 * rep_se_t0 + rep_se_t1 * rep_se_t1), 2)           AS se_rep_change,
  ROUND(sqrt(dem_se_t0 * dem_se_t0 + dem_se_t1 * dem_se_t1), 2)           AS se_dem_change,
  ROUND(sqrt(ind_se_t0 * ind_se_t0 + ind_se_t1 * ind_se_t1), 2)           AS se_ind_change,
  ROUND(sqrt(rep_se_t0 * rep_se_t0 + rep_se_t1 * rep_se_t1
           + dem_se_t0 * dem_se_t0 + dem_se_t1 * dem_se_t1), 2)           AS se_winner_minus_loser_gap_change,  -- = SE of the gap change

  -- ── Empirical yardstick: how much the gap drifts when NO federal election falls between waves ─────────
  -- Sampling SEs above are ~1-2 points, but CES quiet years move the Rep-Dem gap by 12-17 points (fieldwork season, news,
  -- question context). That year-to-year drift, not sampling error, is the honest benchmark for an annual comparison.
  -- Same source, window, party definition and measure; NULL where a source has no quiet pairs (ANES, Pew).
  ROUND(AVG(CASE WHEN comparison_type = 'No federal election between waves'
                 THEN abs((rep_t1 - dem_t1) - (rep_t0 - dem_t0)) END)
          OVER (PARTITION BY source, comparison_window, party_definition, measure), 2)   AS quiet_years_mean_abs_gap_change,
  ROUND(MAX(CASE WHEN comparison_type = 'No federal election between waves'
                 THEN abs((rep_t1 - dem_t1) - (rep_t0 - dem_t0)) END)
          OVER (PARTITION BY source, comparison_window, party_definition, measure), 2)   AS quiet_years_max_abs_gap_change,
  COUNT(CASE WHEN comparison_type = 'No federal election between waves' THEN 1 END)
          OVER (PARTITION BY source, comparison_window, party_definition, measure)       AS n_quiet_year_pairs
FROM with_winner;

## 3. `fct_sentiment_monthly` — the lede chart
**Grain:** one row per month × series. Series: Republican, Democrat, Independent (UMich party indices, April 2017 onward),
All respondents (full-sample index, January 2016 onward) and **Republican minus Democrat** (the gap, in index points).

The plan's lede chart is "UMich's monthly party lines from 2017 to this month, with elections and inaugurations marked, and Pew's
polls as the cross-check." This mart is the lines and the markers; the Pew points are `fct_party_ratings WHERE source = 'Pew'`.
* `partisan_role` labels the Republican and Democratic lines In-party / Out-party by the White House party on the 1st of the month
  (so January of an inauguration year still belongs to the outgoing party; the event study uses exact event months instead).
* Months before a series begins are kept as NULL rows so every series spans the same calendar; chart tools skip NULLs.
* `event_marker` names the election, inauguration or midterm in that month, for annotation.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_sentiment_monthly
COMMENT 'Lede chart: UMich consumer sentiment by month for Republicans, Democrats, independents, all respondents and the Rep-minus-Dem gap, with in-party/out-party labels and election, inauguration and midterm markers.'
AS
WITH
series_long AS (
  SELECT s.month, s.year, s.white_house_party, x.series, x.series_order, x.sentiment_index
  FROM int_sentiment_monthly AS s
  LATERAL VIEW stack(5,
      'Republican',                1, s.sentiment_rep,
      'Democrat',                  2, s.sentiment_dem,
      'Independent',               3, s.sentiment_ind,
      'All respondents',           4, s.ics_all,
      'Republican minus Democrat', 5, s.partisan_gap_rep_minus_dem
  ) x AS series, series_order, sentiment_index
),

-- Every dated event that gets a marker on the chart. A month never holds two events (Nov elections, Jan inaugurations).
event_markers AS (
  SELECT election_month     AS month, concat(CAST(election_year AS STRING), ' presidential election') AS event_marker,
         'Presidential election' AS event_marker_type
  FROM ref_events
  UNION ALL
  SELECT inauguration_month, concat(CAST(year(inauguration_date) AS STRING), ' inauguration (', president_name, ')'),
         'Inauguration'
  FROM ref_events
  UNION ALL
  SELECT election_month, concat(CAST(election_year AS STRING), ' midterm (', chambers_flipped, ' to ', winning_party, 's)'),
         'Midterm election'
  FROM ref_midterms
)

SELECT
  l.month,
  l.year,
  l.series,
  l.series_order,                                                   -- legend order
  IF(l.series = 'Republican minus Democrat', 'Gap (index points)', 'Index (1966 = 100)') AS series_unit,
  CASE
    WHEN l.series IN ('Republican', 'Democrat') THEN IF(l.series = l.white_house_party, 'In-party', 'Out-party')
    WHEN l.series = 'Independent'               THEN 'Independent'
    WHEN l.series = 'All respondents'           THEN 'All respondents'
    ELSE 'Gap'
  END                                                               AS partisan_role,
  ROUND(l.sentiment_index, 2)                                       AS sentiment_index,
  l.white_house_party,
  m.event_marker,
  m.event_marker_type
FROM series_long AS l
LEFT JOIN event_markers AS m ON l.month = m.month;

## 4. `fct_event_study` — change from the pre-election baseline, month by month
**Grain:** one row per event × month (−6 to +14 from the election month) × series, **only where a sentiment value exists**.
Series: Republican, Democrat, Independent, All respondents, plus a derived **Winning minus losing party** gap series.

Built on `int_event_panel`, which already has the baseline (mean of months −4..−2, i.e. Jul–Sep) and the change from it. This mart
adds three things the chart and the timing estimand need:

**1 · A noise band (decision 6).** For each series, `noise_sd_level` is the robust standard deviation of a single month's
reading. A change from baseline compares one month with a 3-month mean, so its standard error is `σ · √(1 + 1/3)`;
`noise_band_2se` is twice that, and `is_outside_noise_band` flags changes larger than the band. For the gap series,
`σ_gap = √(σ_winner² + σ_loser²)` (the party subsamples are independent).

**2 · The total swing** = change from baseline at month +8 (six months after the January takeover), the plan's denominator.

**3 · Share of the swing complete** = change from baseline ÷ total swing, in percent, computed **only when it is interpretable**
(`is_share_interpretable`), which takes two pre-specified conditions:
* the total swing is larger than its own noise band (`is_total_swing_distinguishable`), and
* the path did not **reverse direction** (`swing_reversed_direction`): no month between the election and +8 sits beyond the noise
  band on the opposite side of the baseline from where the series ends. Independents in 2024–25 rose about 6 points by January and
  then fell about 7 below baseline by July; a "share complete" for that path would read −76%, which means nothing.

Together these replace the preliminary table's informal "not computed" for independents with a rule written down in advance.
Bootstrap intervals for the shares come from notebook 04.

Economic columns are deliberately **not** carried here (they live in `fct_fundamentals_at_transitions` and
`fct_say_vs_do_monthly`), so nothing in this table repeats across series rows.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_event_study
COMMENT 'Event study: UMich sentiment by party around presidential elections and midterms, months -6 to +14 from the election, change from the Jul-Sep baseline, robust noise band, total swing to six months after takeover and share of swing complete.'
AS
WITH
-- ─────────────────────────────────────────────────────────────────────────────
-- Noise per series (decision 6). Month-to-month changes over each series' full coverage, consecutive months only.
-- ─────────────────────────────────────────────────────────────────────────────
sentiment_long AS (
  SELECT s.month, x.series, x.sentiment_index
  FROM int_sentiment_monthly AS s
  LATERAL VIEW stack(4,
      'Republican',      s.sentiment_rep,
      'Democrat',        s.sentiment_dem,
      'Independent',     s.sentiment_ind,
      'All respondents', s.ics_all
  ) x AS series, sentiment_index
  WHERE x.sentiment_index IS NOT NULL
),

month_to_month AS (
  SELECT
    series,
    sentiment_index - LAG(sentiment_index) OVER (PARTITION BY series ORDER BY month) AS mom_change,
    -- guard: only a true one-month step counts (the series are gap-free, but this makes it explicit)
    add_months(LAG(month) OVER (PARTITION BY series ORDER BY month), 1) = month       AS is_consecutive
  FROM sentiment_long
),

median_change AS (
  SELECT series, percentile(mom_change, 0.5) AS median_mom_change
  FROM month_to_month
  WHERE is_consecutive
  GROUP BY series
),

noise AS (
  SELECT
    c.series,
    COUNT(*)                                                              AS noise_n_changes,
    1.4826 * percentile(abs(c.mom_change - m.median_mom_change), 0.5)    AS noise_sd_mom_change,   -- robust SD of a monthly change
    1.4826 * percentile(abs(c.mom_change - m.median_mom_change), 0.5) / sqrt(2)
                                                                          AS noise_sd_level         -- SD of one month's reading
  FROM month_to_month AS c
  JOIN median_change  AS m ON c.series = m.series
  WHERE c.is_consecutive
  GROUP BY c.series
),

-- ─────────────────────────────────────────────────────────────────────────────
-- Party and full-sample rows from the intermediate panel, with their noise.
-- ─────────────────────────────────────────────────────────────────────────────
party_rows AS (
  SELECT
    p.event_key, p.event_type, p.event_label, p.election_date, p.election_month, p.inauguration_month,
    p.winning_party, p.white_house_party_at_election, p.white_house_changed_parties, p.chambers_flipped,
    p.month, p.months_from_election, p.months_from_inauguration,
    p.series, p.party_role,
    p.sentiment_index, p.baseline_index, p.change_from_baseline,
    n.noise_sd_level
  FROM int_event_panel AS p
  LEFT JOIN noise AS n ON p.series = n.series
  WHERE p.sentiment_index IS NOT NULL
),

-- Derived gap series: winning party minus losing party, same month. Inner join, so it exists only where both parties do.
gap_rows AS (
  SELECT
    w.event_key, w.event_type, w.event_label, w.election_date, w.election_month, w.inauguration_month,
    w.winning_party, w.white_house_party_at_election, w.white_house_changed_parties, w.chambers_flipped,
    w.month, w.months_from_election, w.months_from_inauguration,
    'Winning minus losing party'                                          AS series,
    'Winning minus losing party'                                          AS party_role,
    w.sentiment_index      - l.sentiment_index                            AS sentiment_index,
    w.baseline_index       - l.baseline_index                             AS baseline_index,
    w.change_from_baseline - l.change_from_baseline                       AS change_from_baseline,
    sqrt(w.noise_sd_level * w.noise_sd_level + l.noise_sd_level * l.noise_sd_level) AS noise_sd_level
  FROM party_rows AS w
  JOIN party_rows AS l
    ON  w.event_key = l.event_key AND w.month = l.month
    AND w.party_role = 'Winning party' AND l.party_role = 'Losing party'
),

combined AS (
  SELECT * FROM party_rows
  UNION ALL
  SELECT * FROM gap_rows
),

with_swing AS (
  SELECT
    *,
    -- Total swing: change from baseline at +8 (six months after the January takeover), per event x series.
    MAX(IF(months_from_election = 8, change_from_baseline, NULL)) OVER (PARTITION BY event_key, series) AS total_swing,
    -- SE of (one month - mean of 3 baseline months) = sigma * sqrt(4/3); band = 2 SE.
    2 * noise_sd_level * sqrt(4.0 / 3)                                                                  AS noise_band_2se
  FROM combined
),

with_reversal AS (
  SELECT
    *,
    -- Reversal: some month between the election and +8 sits clearly (beyond the noise band) on the OPPOSITE side of the
    -- baseline from where the series ends up. Example: independents in 2024-25 rose ~6 points by January, then fell ~7
    -- below baseline by July. A "share of the swing complete" is meaningless for a path like that, so it is not computed.
    BOOL_OR(months_from_election BETWEEN 0 AND 8
            AND sign(change_from_baseline) = -sign(total_swing)
            AND abs(change_from_baseline) > noise_band_2se)
      OVER (PARTITION BY event_key, series)                                                             AS swing_reversed_direction
  FROM with_swing
)

SELECT
  event_key,
  event_type,
  event_label,
  election_date,
  election_month,
  inauguration_month,
  winning_party,
  white_house_party_at_election,
  white_house_changed_parties,
  chambers_flipped,
  month,
  months_from_election,
  months_from_inauguration,
  months_from_election = 0                                                AS is_election_month,
  months_from_election = 2                                                AS is_takeover_month,   -- inauguration or new Congress (January)
  series,
  party_role,
  ROUND(sentiment_index, 2)                                               AS sentiment_index,
  ROUND(baseline_index, 2)                                                AS baseline_index,
  ROUND(change_from_baseline, 2)                                          AS change_from_baseline,
  ROUND(noise_sd_level, 2)                                                AS noise_sd_level,
  ROUND(noise_band_2se, 2)                                                AS noise_band_2se,
  abs(change_from_baseline) > noise_band_2se                              AS is_outside_noise_band,
  ROUND(total_swing, 2)                                                   AS total_swing,
  abs(total_swing) > noise_band_2se                                       AS is_total_swing_distinguishable,
  COALESCE(swing_reversed_direction, FALSE)                               AS swing_reversed_direction,
  -- Share is reported only when it means something: the swing is bigger than noise AND it did not reverse direction.
  abs(total_swing) > noise_band_2se AND NOT COALESCE(swing_reversed_direction, FALSE)
                                                                          AS is_share_interpretable,
  CASE WHEN abs(total_swing) > noise_band_2se AND NOT COALESCE(swing_reversed_direction, FALSE)
       THEN ROUND(100 * try_divide(change_from_baseline, total_swing), 1) END AS share_of_swing_complete_pct
FROM with_reversal;

## 5. `fct_swing_timing` — how much of the swing was done by Election Day and by the takeover
**Grain:** one row per event × series (events and series with a Jul–Sep baseline).

The plan's co-primary timing estimand (assumption A8, revision of Oct 22): *the share of each party's total swing (pre-election
level to six months after inauguration) that is complete in the election month and in the inauguration month.* Both dates are
reported for every event, whatever they show, which is the guard against having picked the election date after seeing the data.

* `level_*` columns reproduce the plan's table (Jul–Sep, election month, inauguration month, six months after).
* `pct_swing_done_at_election`, `pct_swing_done_at_takeover`: NULL unless `is_share_interpretable` (swing beyond noise and no
  reversal; see `fct_event_study`).
* `first_month_half_done`: first month (from the election month on) at which at least half the swing is complete; the chart
  annotation for "when opinion flipped".
* Midterms are included with the same checkpoints (+2 = new Congress sworn in) for the placebo; their "swing" is usually inside
  the noise band, so their shares are mostly NULL by design.
* Bootstrap intervals: `fct_swing_timing_bootstrap` (notebook 04), joined on `event_key, series`.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_swing_timing
COMMENT 'Co-primary RQ1 timing estimand: share of each series'' swing (Jul-Sep baseline to six months after takeover) complete at the election month and the takeover (inauguration) month, with levels, total swing and noise threshold.'
AS
SELECT
  event_key,
  event_type,
  event_label,
  winning_party,
  white_house_changed_parties,
  series,
  party_role,
  MAX(baseline_index)                                                          AS baseline_jul_sep,
  MAX(IF(months_from_election = 0, sentiment_index, NULL))                     AS level_election_month,
  MAX(IF(months_from_election = 2, sentiment_index, NULL))                     AS level_takeover_month,          -- January
  MAX(IF(months_from_election = 8, sentiment_index, NULL))                     AS level_six_months_after_takeover,
  MAX(IF(months_from_election = 0, change_from_baseline, NULL))                AS change_at_election_month,
  MAX(IF(months_from_election = 2, change_from_baseline, NULL))                AS change_at_takeover_month,
  MAX(total_swing)                                                             AS total_swing,
  MAX(noise_sd_level)                                                          AS noise_sd_level,
  MAX(noise_band_2se)                                                          AS noise_band_2se,
  BOOL_OR(is_total_swing_distinguishable)                                      AS is_total_swing_distinguishable,
  BOOL_OR(swing_reversed_direction)                                            AS swing_reversed_direction,
  BOOL_OR(is_share_interpretable)                                              AS is_share_interpretable,
  MAX(IF(months_from_election = 0, share_of_swing_complete_pct, NULL))         AS pct_swing_done_at_election,
  MAX(IF(months_from_election = 2, share_of_swing_complete_pct, NULL))         AS pct_swing_done_at_takeover,
  MIN(IF(months_from_election BETWEEN 0 AND 8 AND share_of_swing_complete_pct >= 50, months_from_election, NULL))
                                                                               AS first_month_half_done          -- months from election
FROM fct_event_study
WHERE baseline_index IS NOT NULL
GROUP BY event_key, event_type, event_label, winning_party, white_house_changed_parties, series, party_role;

## 6. `fct_fundamentals_at_transitions` — did the economy move when opinion did?
**Grain:** one row per presidential election (7: five transitions + two re-elections) × measure.

Reproduces and extends the plan's fundamentals table. Fundamentals and perceptions sit in **the same table, at the same
checkpoints**, so the comparison the story makes ("inflation moved 0 to 0.6 points while party sentiment moved 17 to 30") is one
read across a row:

| Checkpoint column | Month | Why |
|---|---|---|
| `sep_before` | election − 2 | the plan's "Sep before election" column |
| `election_month` | election month | |
| `inauguration_month` | January after | |
| `six_months_after` | July after (inauguration + 6) | the plan's "July after" column |

* **Fundamentals:** headline inflation, derived unemployment, misery index, gasoline CPI y/y, real hourly earnings y/y, real PCE y/y.
  Unemployment and the CPI components start in 2016–17, so earlier transitions show inflation only (until UNRATE and older CPI are
  added: listed data gaps).
* **Perceptions:** UMich sentiment for the winning party, losing party, independents and all respondents. A perception row needs
  the September starting point, so party rows exist only for 2020 and 2024 (the party series starts April 2017) and 2016–17 has
  the all-respondents row only.
* `is_covid_transition` flags 2020–21, where fundamentals moved for reasons unrelated to the election (plan: "reported but flagged").

In [0]:
%sql
CREATE OR REPLACE TABLE fct_fundamentals_at_transitions
COMMENT 'For every presidential election 2000-2024: inflation, unemployment and other fundamentals next to UMich sentiment by party, at Sep before, election month, inauguration month and July after, with changes.'
AS
WITH
fundamentals AS (
  SELECT
    p.event_key, p.event_label, p.election_date, p.winning_party, p.white_house_party_at_election, p.white_house_changed_parties,
    p.months_from_election, p.months_from_inauguration,
    m.measure_group, m.measure, m.measure_order, m.unit, m.value
  FROM int_event_panel AS p
  LATERAL VIEW stack(6,
      'Fundamentals', 'Inflation (CPI, % y/y)',              1, 'percent', p.inflation_yoy_pct,
      'Fundamentals', 'Unemployment rate (derived, %)',      2, 'percent', p.unemployment_rate_derived_pct,
      'Fundamentals', 'Misery index (%)',                    3, 'percent', p.misery_index_pct,
      'Fundamentals', 'Gasoline CPI (% y/y)',                4, 'percent', p.cpi_gasoline_yoy_pct,
      'Fundamentals', 'Real avg hourly earnings (% y/y)',    5, 'percent', p.real_avg_hourly_earnings_yoy_pct,
      'Fundamentals', 'Real consumer spending, PCE (% y/y)', 6, 'percent', p.pce_real_yoy_pct
  ) m AS measure_group, measure, measure_order, unit, value
  WHERE p.event_type = 'Presidential election'
    AND p.series = 'All respondents'            -- economic columns repeat across the four series rows; read them once
),

perceptions AS (
  SELECT
    event_key, event_label, election_date, winning_party, white_house_party_at_election, white_house_changed_parties,
    months_from_election, months_from_inauguration,
    'Perception'                                                                  AS measure_group,
    CASE party_role
      WHEN 'Winning party'   THEN 'Sentiment: winning party (UMich index)'
      WHEN 'Losing party'    THEN 'Sentiment: losing party (UMich index)'
      WHEN 'Independent'     THEN 'Sentiment: independents (UMich index)'
      WHEN 'All respondents' THEN 'Sentiment: all respondents (UMich index)'
    END                                                                           AS measure,
    CASE party_role WHEN 'Winning party' THEN 7 WHEN 'Losing party' THEN 8 WHEN 'Independent' THEN 9 ELSE 10 END AS measure_order,
    'index (1966 = 100)'                                                          AS unit,
    sentiment_index                                                               AS value
  FROM int_event_panel
  WHERE event_type = 'Presidential election'
),

stacked AS (
  SELECT * FROM fundamentals
  UNION ALL
  SELECT * FROM perceptions
),

checkpoints AS (
  SELECT
    event_key, event_label, election_date, winning_party, white_house_party_at_election, white_house_changed_parties,
    measure_group, measure, measure_order, unit,
    MAX(IF(months_from_election = -2,    value, NULL))        AS sep_before,
    MAX(IF(months_from_election =  0,    value, NULL))        AS election_month,
    MAX(IF(months_from_inauguration = 0, value, NULL))        AS inauguration_month,
    MAX(IF(months_from_inauguration = 6, value, NULL))        AS six_months_after
  FROM stacked
  GROUP BY event_key, event_label, election_date, winning_party, white_house_party_at_election, white_house_changed_parties,
           measure_group, measure, measure_order, unit
)

SELECT
  event_key,
  event_label,
  election_date,
  winning_party,
  white_house_party_at_election                                          AS outgoing_party,
  white_house_changed_parties,
  event_key = 'pres-2020'                                                AS is_covid_transition,
  measure_group,
  measure,
  measure_order,
  unit,
  ROUND(sep_before, 2)                                                   AS sep_before,
  ROUND(election_month, 2)                                               AS election_month,
  ROUND(inauguration_month, 2)                                           AS inauguration_month,
  ROUND(six_months_after, 2)                                             AS six_months_after,
  ROUND(election_month     - sep_before, 2)                              AS change_sep_to_election,
  ROUND(inauguration_month - sep_before, 2)                              AS change_sep_to_inauguration,   -- the story's key comparison
  ROUND(six_months_after   - sep_before, 2)                              AS change_sep_to_six_months_after
FROM checkpoints
-- Fundamentals: keep any row with data at some checkpoint (e.g. 2000-01 has inflation but no unemployment).
-- Perceptions: require the September starting point, otherwise there is no "before" to compare with
-- (the party series starts April 2017, so 2016-17 party rows would show only a July 2017 level).
WHERE (measure_group = 'Fundamentals' AND COALESCE(sep_before, election_month, inauguration_month, six_months_after) IS NOT NULL)
   OR (measure_group = 'Perception'   AND sep_before IS NOT NULL);

## 7. `fct_midterm_placebo` — does opinion follow Congress too?
**Grain:** one row per source × event × checkpoint.

**Logic of the placebo.** If economic ratings track *power* in general, the party that wins a chamber of Congress should gain
relative to the party that lost it, just as the White House winner does. If they track *the presidency* specifically, midterms
should barely move the gap. Every midterm since 2006 was won by the party **out** of the White House, so the placebo always tests
the out-party gaining some power.

**Estimand: change in the winning-minus-losing gap (decision 8).** Intermediate finding 1 showed that three months after the
2022 midterm all three groups had risen 8–11 points together (a common shock), while the Rep–Dem gap barely moved. Party levels
therefore mislead; the gap is the comparison that nets out anything that hits everyone. Presidential elections are included with
identical columns so the chart is midterms vs presidential elections, side by side.

| Source | Events | Checkpoints | Noise yardstick |
|---|---|---|---|
| UMich (monthly) | midterms 2018, 2022; presidential 2020, 2024 | +0, +2, +3, +8 months from the election (decision 7) | `gap_change_noise_band` = 2 × σ_gap × √(4/3), from `fct_event_study` |
| CES (annual) | midterms 2018→19, 2022→23; transitions 2016→17, 2020→21, 2024→25 | next annual wave | 2 × approximate SE of the gap change **and**, stricter, the largest quiet-year gap drift (`quiet_years_max_abs_gap_change`) |

**Read the CES rows with the quiet-year yardstick, not the sampling band.** CES sampling SEs make almost any annual change look
"significant", but the Rep–Dem gap drifts 12–17 points between CES waves even with no election in between. The CES midterm
comparisons (2018→19: +37; 2022→23: −12.5, i.e. the *loser* gained) are in that drift range or driven by the White House party's own
voters moving, so the **UMich monthly placebo is the primary placebo** and CES is supporting context. Presidential transitions move
the CES gap 77–95 points, well beyond any quiet year.

`rep_minus_dem_gap_change` is kept for reference (the intermediate handoff quoted it: 2022 +2.2, 2018 −3.1 at +3 months).

In [0]:
%sql
CREATE OR REPLACE TABLE fct_midterm_placebo
COMMENT 'Midterm placebo (RQ1 validity check): change in the winning-minus-losing party gap after midterms vs presidential elections, UMich monthly checkpoints and CES next-wave comparisons, with noise yardsticks.'
AS
WITH
umich AS (
  SELECT
    'UMich (monthly)'                                                                        AS source,
    event_key, event_type, event_label, winning_party, chambers_flipped,
    months_from_election                                                                     AS checkpoint_months_from_election,
    CASE months_from_election
      WHEN 0 THEN 'Election month'
      WHEN 2 THEN 'Takeover month (inauguration or new Congress)'
      WHEN 3 THEN 'Three months after election'
      WHEN 8 THEN 'Six months after takeover'
    END                                                                                      AS checkpoint_label,
    MAX(IF(series = 'Winning minus losing party', change_from_baseline, NULL))               AS winner_minus_loser_gap_change,
    MAX(IF(series = 'Winning minus losing party', noise_band_2se, NULL))                     AS gap_change_noise_band,
    MAX(IF(party_role = 'Winning party', change_from_baseline, NULL))                        AS winning_party_change,
    MAX(IF(party_role = 'Losing party',  change_from_baseline, NULL))                        AS losing_party_change,
    MAX(IF(series = 'Independent',       change_from_baseline, NULL))                        AS independent_change,
    MAX(IF(series = 'All respondents',   change_from_baseline, NULL))                        AS all_respondents_change,
    MAX(IF(series = 'Republican', change_from_baseline, NULL))
      - MAX(IF(series = 'Democrat', change_from_baseline, NULL))                             AS rep_minus_dem_gap_change,
    CAST(NULL AS DOUBLE)                                                                     AS quiet_years_mean_abs_gap_change,  -- CES only
    CAST(NULL AS DOUBLE)                                                                     AS quiet_years_max_abs_gap_change,
    'UMich index points'                                                                     AS unit
  FROM fct_event_study
  WHERE months_from_election IN (0, 2, 3, 8)
    AND baseline_index IS NOT NULL
  GROUP BY event_key, event_type, event_label, winning_party, chambers_flipped, months_from_election
  -- keep events the party series covers (the gap series needs both parties)
  HAVING MAX(IF(series = 'Winning minus losing party', change_from_baseline, NULL)) IS NOT NULL
),

ces AS (
  SELECT
    'CES (annual)'                                                                           AS source,
    c.event_key, c.event_type, c.event_label, c.winning_party,
    m.chambers_flipped,
    c.months_from_election_to_t1                                                             AS checkpoint_months_from_election,
    concat('Next annual wave (', c.wave_key_t0, ' to ', c.wave_key_t1, ')')                  AS checkpoint_label,
    c.winner_minus_loser_gap_change,
    2 * c.se_winner_minus_loser_gap_change                                                   AS gap_change_noise_band,
    c.winning_party_change,
    c.losing_party_change,
    c.ind_change                                                                             AS independent_change,
    c.all_change                                                                             AS all_respondents_change,
    c.rep_minus_dem_gap_change,
    c.quiet_years_mean_abs_gap_change,
    c.quiet_years_max_abs_gap_change,
    'percentage points (% better, main spec)'                                                AS unit
  FROM fct_wave_contrasts AS c
  LEFT JOIN ref_midterms AS m ON c.event_key = concat('mid-', CAST(m.election_year AS STRING))
  WHERE c.source = 'CES'
    AND c.measure = 'pct_better'
    AND c.party_definition = 'Leaners with parties'
    AND c.comparison_type IN ('Presidential transition', 'Midterm (a chamber of Congress changed hands)')
)

SELECT
  source,
  event_key,
  event_type,
  event_label,
  event_type = 'Midterm election'                                         AS is_placebo,
  winning_party,
  chambers_flipped,
  checkpoint_months_from_election,
  checkpoint_label,
  unit,
  ROUND(winner_minus_loser_gap_change, 2)                                 AS winner_minus_loser_gap_change,
  ROUND(gap_change_noise_band, 2)                                         AS gap_change_noise_band,
  abs(winner_minus_loser_gap_change) > gap_change_noise_band              AS is_gap_change_outside_noise,
  -- CES only: the stricter, empirical yardstick (largest gap drift in a year with no federal election between waves)
  quiet_years_mean_abs_gap_change,
  quiet_years_max_abs_gap_change,
  abs(winner_minus_loser_gap_change) > quiet_years_max_abs_gap_change     AS is_gap_change_beyond_quiet_years,
  ROUND(winning_party_change, 2)                                          AS winning_party_change,
  ROUND(losing_party_change, 2)                                           AS losing_party_change,
  ROUND(independent_change, 2)                                            AS independent_change,
  ROUND(all_respondents_change, 2)                                        AS all_respondents_change,
  ROUND(rep_minus_dem_gap_change, 2)                                      AS rep_minus_dem_gap_change
FROM (SELECT * FROM umich UNION ALL SELECT * FROM ces);

## 8. `fct_pocketbook_gap` — your wallet vs "the economy" (RQ3)
**Grain:** one row per ANES survey year (2016, 2020, 2024) × outcome spec × party definition.

**Estimand (plan):** for each year, the partisan gap on the national economy and on own finances, each = % saying "worse" among
the **out-party** minus the **in-party**; reported as a difference and a ratio.

* **Two outcome specs** (decision 1): *Main: do not know excluded* and *Robustness: do not know in base* (what the plan's preliminary
  table used; intermediate QA showed the choice barely matters for ANES).
* **Two party definitions** (decision 2).
* **Two ratios**, because the plan uses both directions: `national_to_personal_ratio` (the preliminary table's "2.0×") and
  `personal_to_national_ratio` (the RQ table's "personal ÷ national"; 1 = equally partisan, 0 = wallet not partisan at all).
* **Net-rating gaps** as a robustness view: in-party net rating minus out-party net rating (positive = in-party rosier).
* **Uncertainty.** `*_gap_se_approx` combines the two parties' Kish SEs. `gap_difference_se_conservative` ignores the positive
  correlation between a respondent's two answers, so it overstates the SE. The reported interval for the ratio is the respondent
  bootstrap in `fct_pocketbook_gap_bootstrap` (notebook 04). Design-based SEs need strata/PSU from the yearly study files (data gap).
* Gaps and their difference are stored to **3 decimals** (shares to 2) so that rounding to one decimal for display does not
  round twice: the 2020 own-finances gap is 11.046, which is 11.0, but would show as 11.1 if stored as 11.05.
* In-party uses an assumed 1 October fieldwork date for ANES (fall pre-election wave), flagged upstream. Interview mode differs
  across years (2016 mixed, 2020 mostly web); a mode-specific cut is a filter on `int_respondents.interview_mode`.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_pocketbook_gap
COMMENT 'RQ3: ANES partisan gap (out-party minus in-party % worse) on the national economy vs own finances, 2016/2020/2024, main and robustness specs, difference, both ratios, net-rating gaps, approximate SEs.'
AS
WITH
-- One row per respondent x party definition x outcome spec. Two chained stack()s keep all four combinations in one pass.
base AS (
  SELECT
    r.survey_year,
    r.weight,
    r.white_house_party,
    d.party_definition,
    d.partisan_role,
    s.outcome_spec,
    s.national_worse,
    s.personal_worse,
    s.national_net,
    s.personal_net
  FROM int_respondents AS r
  LATERAL VIEW stack(2,
      'Leaners with parties',    r.partisan_role,
      'Leaners as independents', r.partisan_role_strict
  ) d AS party_definition, partisan_role
  LATERAL VIEW stack(2,
      'Main: do not know excluded',
          r.econ_worse,            r.personal_worse,
          r.econ_net,              r.personal_net,
      'Robustness: do not know in base',
          r.econ_worse_ns_as_same, r.personal_worse_ns_as_same,
          r.econ_better_ns_as_same - r.econ_worse_ns_as_same,                 -- net with "do not know" counted as 0
          r.personal_better_ns_as_same - r.personal_worse_ns_as_same
  ) s AS outcome_spec, national_worse, personal_worse, national_net, personal_net
  WHERE r.source = 'ANES'
    AND d.partisan_role IN ('In-party', 'Out-party')
),

by_role AS (
  SELECT
    survey_year, party_definition, outcome_spec, partisan_role,
    MAX(white_house_party)                                                                                   AS white_house_party,
    100 * SUM(national_worse * weight) / NULLIF(SUM(IF(national_worse IS NOT NULL, weight, 0)), 0)           AS national_pct_worse,
    100 * SUM(personal_worse * weight) / NULLIF(SUM(IF(personal_worse IS NOT NULL, weight, 0)), 0)           AS personal_pct_worse,
    100 * SUM(national_net   * weight) / NULLIF(SUM(IF(national_net   IS NOT NULL, weight, 0)), 0)           AS national_net_rating,
    100 * SUM(personal_net   * weight) / NULLIF(SUM(IF(personal_net   IS NOT NULL, weight, 0)), 0)           AS personal_net_rating,
    COUNT(national_worse)                                                                                    AS national_n,
    COUNT(personal_worse)                                                                                    AS personal_n,
    POWER(SUM(IF(national_worse IS NOT NULL, weight, 0)), 2)
      / NULLIF(SUM(IF(national_worse IS NOT NULL, weight * weight, 0)), 0)                                   AS national_n_eff,
    POWER(SUM(IF(personal_worse IS NOT NULL, weight, 0)), 2)
      / NULLIF(SUM(IF(personal_worse IS NOT NULL, weight * weight, 0)), 0)                                   AS personal_n_eff
  FROM base
  GROUP BY survey_year, party_definition, outcome_spec, partisan_role
),

with_se AS (
  SELECT
    *,
    100 * sqrt((national_pct_worse / 100) * (1 - national_pct_worse / 100) / national_n_eff) AS national_se,
    100 * sqrt((personal_pct_worse / 100) * (1 - personal_pct_worse / 100) / personal_n_eff) AS personal_se
  FROM by_role
),

-- In-party and out-party side by side.
wide AS (
  SELECT
    survey_year, party_definition, outcome_spec,
    MAX(white_house_party)                                                         AS white_house_party,
    MAX(IF(partisan_role = 'In-party',  national_pct_worse,  NULL))                AS national_pct_worse_in_party,
    MAX(IF(partisan_role = 'Out-party', national_pct_worse,  NULL))                AS national_pct_worse_out_party,
    MAX(IF(partisan_role = 'In-party',  personal_pct_worse,  NULL))                AS personal_pct_worse_in_party,
    MAX(IF(partisan_role = 'Out-party', personal_pct_worse,  NULL))                AS personal_pct_worse_out_party,
    MAX(IF(partisan_role = 'In-party',  national_net_rating, NULL))                AS national_net_in_party,
    MAX(IF(partisan_role = 'Out-party', national_net_rating, NULL))                AS national_net_out_party,
    MAX(IF(partisan_role = 'In-party',  personal_net_rating, NULL))                AS personal_net_in_party,
    MAX(IF(partisan_role = 'Out-party', personal_net_rating, NULL))                AS personal_net_out_party,
    MAX(IF(partisan_role = 'In-party',  national_se, NULL))                        AS national_se_in,
    MAX(IF(partisan_role = 'Out-party', national_se, NULL))                        AS national_se_out,
    MAX(IF(partisan_role = 'In-party',  personal_se, NULL))                        AS personal_se_in,
    MAX(IF(partisan_role = 'Out-party', personal_se, NULL))                        AS personal_se_out,
    MAX(IF(partisan_role = 'In-party',  national_n, NULL))                         AS national_n_in_party,
    MAX(IF(partisan_role = 'Out-party', national_n, NULL))                         AS national_n_out_party,
    MAX(IF(partisan_role = 'In-party',  personal_n, NULL))                         AS personal_n_in_party,
    MAX(IF(partisan_role = 'Out-party', personal_n, NULL))                         AS personal_n_out_party
  FROM with_se
  GROUP BY survey_year, party_definition, outcome_spec
),

gaps AS (
  SELECT
    *,
    national_pct_worse_out_party - national_pct_worse_in_party                     AS national_gap,
    personal_pct_worse_out_party - personal_pct_worse_in_party                     AS personal_gap,
    sqrt(national_se_in * national_se_in + national_se_out * national_se_out)      AS national_gap_se,
    sqrt(personal_se_in * personal_se_in + personal_se_out * personal_se_out)      AS personal_gap_se
  FROM wide
)

SELECT
  survey_year,
  white_house_party,
  party_definition,
  outcome_spec,
  outcome_spec LIKE 'Main%' AND party_definition = 'Leaners with parties'          AS is_main_spec,

  -- ── National economy ───────────────────────────────────────────────────────
  ROUND(national_pct_worse_in_party, 2)                                            AS national_pct_worse_in_party,
  ROUND(national_pct_worse_out_party, 2)                                           AS national_pct_worse_out_party,
  ROUND(national_gap, 3)                                                           AS national_gap_pts,
  ROUND(national_gap_se, 2)                                                        AS national_gap_se_approx,

  -- ── Own finances ───────────────────────────────────────────────────────────
  ROUND(personal_pct_worse_in_party, 2)                                            AS personal_pct_worse_in_party,
  ROUND(personal_pct_worse_out_party, 2)                                           AS personal_pct_worse_out_party,
  ROUND(personal_gap, 3)                                                           AS personal_gap_pts,
  ROUND(personal_gap_se, 2)                                                        AS personal_gap_se_approx,

  -- ── The RQ3 comparison ─────────────────────────────────────────────────────
  ROUND(national_gap - personal_gap, 3)                                            AS gap_difference_pts,          -- > 0: national more partisan
  ROUND(sqrt(national_gap_se * national_gap_se + personal_gap_se * personal_gap_se), 2)
                                                                                   AS gap_difference_se_conservative,
  ROUND(try_divide(national_gap, personal_gap), 2)                                 AS national_to_personal_ratio,  -- plan table: "2.0x"
  ROUND(try_divide(personal_gap, national_gap), 3)                                 AS personal_to_national_ratio,  -- RQ table: personal / national

  -- ── Robustness view: net-rating gaps (in-party minus out-party) ────────────
  ROUND(national_net_in_party - national_net_out_party, 2)                         AS national_net_gap_pts,
  ROUND(personal_net_in_party - personal_net_out_party, 2)                         AS personal_net_gap_pts,

  -- ── Sample sizes ───────────────────────────────────────────────────────────
  national_n_in_party,
  national_n_out_party,
  personal_n_in_party,
  personal_n_out_party
FROM gaps;

## 9. `fct_expected_vs_actual_inflation` — what people expect prices to do vs what they did (RQ2)
**Grain:** one row per month, January 2016 to the last UMich month.

* **Primary measure:** median year-ahead expectation (`px1_med`), because a few extreme answers pull the mean above actual
  inflation in most months (plan's tradeoff table). Mean is carried too.
* **Two comparisons, named for what they compare** (from `int_sentiment_monthly`): `expected_minus_current_inflation_pct`
  (vs the past 12 months; the plan's measure) and `forecast_error_12m_pct` (vs the following 12 months; "were people right?").
* **The salience test (one pre-specified contrast):** expected inflation minus gasoline, food-at-home and core CPI y/y. If the
  gap to the *visible* prices is consistently smaller than the gap to core, visible prices explain expectations better. CPI component
  y/y starts January 2017 (intermediate finding 3).
* **Demographic subgroups** (age, income terciles) are read from `stg_umich_index` directly, as the intermediate notebook
  intended (subgroup series were not widened upstream). UMich has **no expectations by party** in the repo: the biggest open
  question for the 2025 gap (plan: "the question is whether it is partisan") stays a listed data gap.
* Gas expectations stay in **cents per gallon** (no pump price in the repo; FRED GASREGW would close this gap).

In [0]:
%sql
CREATE OR REPLACE TABLE fct_expected_vs_actual_inflation
COMMENT 'RQ2: UMich year-ahead inflation expectations (median primary, mean, 5-year, age and income subgroups) vs actual headline inflation (past and following 12 months) and CPI components (gas, food at home, core, shelter), monthly 2016 onward.'
AS
WITH
subgroups AS (
  -- px1_med for the six demographic subgroups the plan names (age and income), month x subgroup -> one row per month
  SELECT
    month,
    MAX(IF(subgroup_code = 'a1834', value, NULL))  AS expected_inflation_1y_median_age_18_34_pct,
    MAX(IF(subgroup_code = 'a3554', value, NULL))  AS expected_inflation_1y_median_age_35_54_pct,
    MAX(IF(subgroup_code = 'a5597', value, NULL))  AS expected_inflation_1y_median_age_55_plus_pct,
    MAX(IF(subgroup_code = 'y13',   value, NULL))  AS expected_inflation_1y_median_income_bottom_third_pct,
    MAX(IF(subgroup_code = 'y23',   value, NULL))  AS expected_inflation_1y_median_income_middle_third_pct,
    MAX(IF(subgroup_code = 'y33',   value, NULL))  AS expected_inflation_1y_median_income_top_third_pct
  FROM stg_umich_index
  WHERE measure_code = 'px1_med'
  GROUP BY month
)

SELECT
  -- ── Calendar ───────────────────────────────────────────────────────────────
  s.month,
  s.year,
  s.white_house_party,
  e.is_shutdown_gap_month,                                               -- Oct 2025: no CPI, all comparisons NULL

  -- ── Expectations (percent) ─────────────────────────────────────────────────
  s.expected_inflation_1y_median_pct,                                    -- primary
  s.expected_inflation_1y_mean_pct,
  s.expected_inflation_5y_median_pct,
  s.expected_gas_change_1y_median_cents,                                 -- cents per gallon, NOT percent

  -- ── Actual inflation ───────────────────────────────────────────────────────
  s.actual_inflation_current_pct,                                        -- headline CPI, past 12 months
  s.actual_inflation_next_12m_pct,                                       -- headline CPI, the 12 months the expectation was about
  e.cpi_core_yoy_pct,
  e.cpi_gasoline_yoy_pct,
  e.cpi_food_at_home_yoy_pct,
  e.cpi_shelter_yoy_pct,

  -- ── Gaps ───────────────────────────────────────────────────────────────────
  s.expected_minus_current_inflation_pct,                                -- plan's measure (median)
  s.expected_mean_minus_current_inflation_pct,
  s.forecast_error_12m_pct,                                              -- median vs realized next-12-month inflation
  s.expected_inflation_1y_median_pct > s.actual_inflation_current_pct    AS median_expectation_above_actual,
  s.expected_inflation_1y_mean_pct   > s.actual_inflation_current_pct    AS mean_expectation_above_actual,
  -- Salience test (pre-specified): which actual series is the expectation closest to?
  ROUND(s.expected_inflation_1y_median_pct - e.cpi_gasoline_yoy_pct, 2)     AS expected_minus_gasoline_yoy_pct,
  ROUND(s.expected_inflation_1y_median_pct - e.cpi_food_at_home_yoy_pct, 2) AS expected_minus_food_at_home_yoy_pct,
  ROUND(s.expected_inflation_1y_median_pct - e.cpi_core_yoy_pct, 2)         AS expected_minus_core_yoy_pct,

  -- ── Subgroups ──────────────────────────────────────────────────────────────
  g.expected_inflation_1y_median_age_18_34_pct,
  g.expected_inflation_1y_median_age_35_54_pct,
  g.expected_inflation_1y_median_age_55_plus_pct,
  g.expected_inflation_1y_median_income_bottom_third_pct,
  g.expected_inflation_1y_median_income_middle_third_pct,
  g.expected_inflation_1y_median_income_top_third_pct
FROM int_sentiment_monthly AS s
LEFT JOIN int_econ_monthly AS e ON s.month = e.month
LEFT JOIN subgroups        AS g ON s.month = g.month;

## 10. `fct_expected_vs_actual_inflation_by_year` — the yearly numbers quoted in the plan
**Grain:** one row per calendar year, plus one **All months** row (`GROUPING SETS`).
Averages skip NULL months (e.g. October 2025); `n_months_compared` says how many months each average contains, so 2026 (a
partial year) and 2016 (no CPI components) are visibly thinner. The salience columns show the average absolute gap to each
price series: the smallest is the series expectations track most closely that year.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_expected_vs_actual_inflation_by_year
COMMENT 'RQ2 yearly summary: average expected minus actual inflation (median and mean), forecast error, peak expectation, share of months expectations exceeded actual, and average absolute gap to gas, food-at-home and core CPI. One row per year plus an all-months row.'
AS
SELECT
  COALESCE(CAST(year AS STRING), 'All months')                                       AS period,
  year,                                                                              -- NULL on the all-months row
  COUNT(expected_minus_current_inflation_pct)                                        AS n_months_compared,
  ROUND(AVG(expected_inflation_1y_median_pct), 2)                                    AS avg_expected_median_pct,
  ROUND(AVG(IF(expected_minus_current_inflation_pct IS NOT NULL, actual_inflation_current_pct, NULL)), 2)
                                                                                     AS avg_actual_inflation_pct,
  ROUND(AVG(expected_minus_current_inflation_pct), 2)                                AS avg_expected_minus_current_pct,
  ROUND(AVG(expected_mean_minus_current_inflation_pct), 2)                           AS avg_expected_mean_minus_current_pct,
  ROUND(AVG(forecast_error_12m_pct), 2)                                              AS avg_forecast_error_12m_pct,
  COUNT(forecast_error_12m_pct)                                                      AS n_months_with_forecast_error,
  MAX(expected_inflation_1y_median_pct)                                              AS peak_expected_median_pct,
  MAX_BY(month, expected_inflation_1y_median_pct)                                    AS peak_expected_median_month,
  ROUND(100 * AVG(CAST(median_expectation_above_actual AS INT)), 0)                  AS pct_months_median_above_actual,
  ROUND(100 * AVG(CAST(mean_expectation_above_actual   AS INT)), 0)                  AS pct_months_mean_above_actual,
  ROUND(AVG(abs(expected_minus_gasoline_yoy_pct)), 2)                                AS avg_abs_gap_to_gasoline_pct,
  ROUND(AVG(abs(expected_minus_food_at_home_yoy_pct)), 2)                            AS avg_abs_gap_to_food_at_home_pct,
  ROUND(AVG(abs(expected_minus_core_yoy_pct)), 2)                                    AS avg_abs_gap_to_core_pct,
  ROUND(AVG(abs(expected_minus_current_inflation_pct)), 2)                           AS avg_abs_gap_to_headline_pct
FROM fct_expected_vs_actual_inflation
GROUP BY GROUPING SETS ((year), ());

## 11. `fct_rq2_annual_panel` — the "thermometer" next to the fundamentals
**Grain:** one row per CES survey year × economic window length (1, 3 or 6 months; 3 is primary, A5).

RQ2 asks whether the **non-partisan component** of perception tracks fundamentals. This mart puts the two non-partisan
estimates SQL can produce next to the economy each wave had just lived through:
* independents' % better and net rating (A2), and the all-respondents net rating, with their year-over-year **changes**;
* fundamentals averaged over the trailing window from `int_econ_windows`, with their changes vs the previous wave.

The third estimate, the **year effects** α<sub>t</sub> from the RQ1 model, comes from notebook 04 (`fct_rq1_year_effects`), which
joins it here on `survey_year` and computes the correlations with bootstrap intervals (`fct_rq2_correlations`).

**Power warning (plan):** 10 waves give 9 changes. Correlations are descriptive; no window or component is chosen by searching
(all three windows are in the table so none is hidden). **CES 2025 window caveat** (intermediate finding 2): the 1-month window is
empty and the 3-month window averages 2 months because it touches October 2025; `is_complete_window` and `n_months_*` show it.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_rq2_annual_panel
COMMENT 'RQ2 annual panel: CES independents'' and all respondents'' economic ratings (levels and year-over-year changes) next to fundamentals in the 1/3/6-month window before fieldwork (levels and changes vs previous wave), one row per CES year x window.'
AS
WITH
perceptions AS (
  SELECT
    survey_year,
    MAX(IF(party_group = 'Independent',     pct_better, NULL))                           AS ind_pct_better,
    MAX(IF(party_group = 'Independent',     net_rating, NULL))                           AS ind_net_rating,
    MAX(IF(party_group = 'Independent',     se_net_rating_approx, NULL))                 AS ind_net_rating_se_approx,
    MAX(IF(party_group = 'All respondents', net_rating, NULL))                           AS all_net_rating,
    MAX(IF(party_group = 'Republican', pct_better, NULL))
      - MAX(IF(party_group = 'Democrat', pct_better, NULL))                              AS rep_minus_dem_gap_pct_better
  FROM fct_party_ratings
  WHERE source = 'CES' AND party_definition = 'Leaners with parties'
  GROUP BY survey_year
)

SELECT
  w.survey_year,
  w.window_months,
  w.is_primary_window,
  w.fieldwork_month,
  w.window_start_month,
  w.window_end_month,
  w.n_months_inflation,
  w.n_months_unemployment,
  w.n_months_cpi_components,
  w.is_complete_window,

  -- ── Perceptions (the "thermometer") ────────────────────────────────────────
  p.ind_pct_better,
  p.ind_net_rating,
  p.ind_net_rating_se_approx,
  p.all_net_rating,
  p.rep_minus_dem_gap_pct_better,
  ROUND(p.ind_pct_better - LAG(p.ind_pct_better) OVER prev, 2)                          AS ind_pct_better_chg,
  ROUND(p.ind_net_rating - LAG(p.ind_net_rating) OVER prev, 2)                          AS ind_net_rating_chg,
  ROUND(p.all_net_rating - LAG(p.all_net_rating) OVER prev, 2)                          AS all_net_rating_chg,

  -- ── Fundamentals in the window (levels) ────────────────────────────────────
  w.inflation_yoy_pct,
  w.unemployment_rate_pct,
  w.misery_index_pct,
  w.cpi_core_yoy_pct,
  w.cpi_gasoline_yoy_pct,
  w.cpi_food_at_home_yoy_pct,
  w.cpi_shelter_yoy_pct,
  w.real_avg_hourly_earnings_yoy_pct,

  -- ── Fundamentals: change vs previous wave (computed upstream where available, here for core and shelter) ──
  w.inflation_chg_vs_prev_wave,
  w.unemployment_chg_vs_prev_wave,
  w.misery_chg_vs_prev_wave,
  w.gasoline_yoy_chg_vs_prev_wave,
  w.food_at_home_yoy_chg_vs_prev_wave,
  w.real_earnings_yoy_chg_vs_prev_wave,
  ROUND(w.cpi_core_yoy_pct    - LAG(w.cpi_core_yoy_pct)    OVER prev, 2)                AS core_yoy_chg_vs_prev_wave,
  ROUND(w.cpi_shelter_yoy_pct - LAG(w.cpi_shelter_yoy_pct) OVER prev, 2)                AS shelter_yoy_chg_vs_prev_wave
FROM int_econ_windows AS w
JOIN perceptions      AS p ON w.survey_year = p.survey_year
WHERE w.source = 'CES'
WINDOW prev AS (PARTITION BY w.window_months ORDER BY w.survey_year);

## 12. `fct_say_vs_do_monthly` — what people say vs what households do (RQ4, aggregate)
**Grain:** one row per month, January 2016 to the last UMich month.

Party sentiment next to real consumer spending (PCE), the saving rate and debt service, with month-over-month changes and an
event clock (`months_since_presidential_election`) so the same table draws the plan's "say vs do" event plots and the monthly
correlation the preliminary results quoted (|r| ≤ 0.16 between monthly changes in party sentiment and real PCE growth).

**Framing (plan):** aggregate spending cannot separate partisans, so this is context, not a test of whether Republicans and Democrats
spend differently. That micro question is the Imprint extension (ZIP-level card spend by ZIP partisanship).
* Debt service is quarterly (release months only, never filled).
* BEA revises PCE and the saving rate: record the download vintage in `METHODS.md`.

In [0]:
%sql
CREATE OR REPLACE TABLE fct_say_vs_do_monthly
COMMENT 'RQ4 aggregate: monthly UMich sentiment by party (levels and month-over-month changes) next to real PCE growth, saving rate and debt service, with months since the last presidential election and event markers.'
AS
WITH
last_election AS (
  SELECT s.month, MAX(e.election_month) AS last_presidential_election_month
  FROM int_sentiment_monthly AS s
  JOIN ref_events AS e ON e.election_month <= s.month
  GROUP BY s.month
),

event_markers AS (
  SELECT election_month AS month, concat(CAST(election_year AS STRING), ' presidential election') AS event_marker FROM ref_events
  UNION ALL
  SELECT inauguration_month, concat(CAST(year(inauguration_date) AS STRING), ' inauguration')          FROM ref_events
  UNION ALL
  SELECT election_month, concat(CAST(election_year AS STRING), ' midterm')                             FROM ref_midterms
)

SELECT
  -- ── Calendar and event clock ───────────────────────────────────────────────
  s.month,
  s.year,
  s.white_house_party,
  CAST(months_between(s.month, le.last_presidential_election_month) AS INT)        AS months_since_presidential_election,
  m.event_marker,

  -- ── Say: sentiment levels ──────────────────────────────────────────────────
  s.sentiment_rep,
  s.sentiment_dem,
  s.sentiment_ind,
  s.ics_all,
  s.partisan_gap_rep_minus_dem,

  -- ── Say: month-over-month changes (int_sentiment_monthly is a gap-free spine, so LAG(1) = previous month) ──
  ROUND(s.sentiment_rep - LAG(s.sentiment_rep) OVER w, 2)                          AS sentiment_rep_mom_chg,
  ROUND(s.sentiment_dem - LAG(s.sentiment_dem) OVER w, 2)                          AS sentiment_dem_mom_chg,
  ROUND(s.sentiment_ind - LAG(s.sentiment_ind) OVER w, 2)                          AS sentiment_ind_mom_chg,
  ROUND(s.ics_all       - LAG(s.ics_all)       OVER w, 2)                          AS ics_all_mom_chg,

  -- ── Do: spending, saving, borrowing ────────────────────────────────────────
  e.pce_real_bn_2017usd,
  e.pce_real_mom_pct,
  e.pce_real_yoy_pct,
  e.personal_saving_rate_pct,
  ROUND(e.personal_saving_rate_pct - LAG(e.personal_saving_rate_pct) OVER w, 2)    AS personal_saving_rate_mom_chg,
  e.debt_service_pct_disposable_income,                                            -- quarterly; release months only

  -- ── Context ────────────────────────────────────────────────────────────────
  e.inflation_yoy_pct,
  e.unemployment_rate_derived_pct
FROM int_sentiment_monthly AS s
LEFT JOIN int_econ_monthly AS e  ON s.month = e.month
LEFT JOIN last_election    AS le ON s.month = le.month
LEFT JOIN event_markers    AS m  ON s.month = m.month
WINDOW w AS (ORDER BY s.month);

## 13. QA checks
Each check either returns 0 failures or reproduces a number already verified upstream (plan, staging or intermediate handoff),
so an aggregation slip shows up as a mismatch. In dbt: grain checks → `unique` / `dbt_utils.unique_combination_of_columns`
tests in `schema.yml`; reconciliations → singular tests in `dbt/tests/`.

**13a · Grain.** Every `failures` value should be **0**.

In [0]:
%sql
SELECT 'fct_party_ratings: duplicate wave/definition/party' AS check,
       COUNT(*) - COUNT(DISTINCT wave_key, party_definition, party_group) AS failures FROM fct_party_ratings
UNION ALL SELECT 'fct_wave_contrasts: duplicate pair/definition/measure',
       COUNT(*) - COUNT(DISTINCT source, comparison_window, wave_key_t0, wave_key_t1, party_definition, measure) FROM fct_wave_contrasts
UNION ALL SELECT 'fct_sentiment_monthly: duplicate month/series',
       COUNT(*) - COUNT(DISTINCT month, series) FROM fct_sentiment_monthly
UNION ALL SELECT 'fct_event_study: duplicate event/month/series',
       COUNT(*) - COUNT(DISTINCT event_key, month, series) FROM fct_event_study
UNION ALL SELECT 'fct_swing_timing: duplicate event/series',
       COUNT(*) - COUNT(DISTINCT event_key, series) FROM fct_swing_timing
UNION ALL SELECT 'fct_fundamentals_at_transitions: duplicate event/measure',
       COUNT(*) - COUNT(DISTINCT event_key, measure) FROM fct_fundamentals_at_transitions
UNION ALL SELECT 'fct_midterm_placebo: duplicate source/event/checkpoint',
       COUNT(*) - COUNT(DISTINCT source, event_key, checkpoint_label) FROM fct_midterm_placebo
UNION ALL SELECT 'fct_pocketbook_gap: duplicate year/spec/definition',
       COUNT(*) - COUNT(DISTINCT survey_year, outcome_spec, party_definition) FROM fct_pocketbook_gap
UNION ALL SELECT 'fct_pocketbook_gap: rows (expect 3 years x 2 specs x 2 definitions = 12) minus 12',
       COUNT(*) - 12 FROM fct_pocketbook_gap
UNION ALL SELECT 'fct_expected_vs_actual_inflation: duplicate month',
       COUNT(*) - COUNT(DISTINCT month) FROM fct_expected_vs_actual_inflation
UNION ALL SELECT 'fct_rq2_annual_panel: duplicate year/window',
       COUNT(*) - COUNT(DISTINCT survey_year, window_months) FROM fct_rq2_annual_panel
UNION ALL SELECT 'fct_say_vs_do_monthly: duplicate month',
       COUNT(*) - COUNT(DISTINCT month) FROM fct_say_vs_do_monthly
UNION ALL SELECT 'fct_party_ratings: share outside 0-100',
       COUNT(*) FROM fct_party_ratings
       WHERE pct_positive NOT BETWEEN 0 AND 100 OR pct_better NOT BETWEEN 0 AND 100 OR pct_worse NOT BETWEEN 0 AND 100
UNION ALL SELECT 'fct_wave_contrasts: gap change <> winner change - loser change',
       COUNT(*) FROM fct_wave_contrasts
       WHERE abs(winner_minus_loser_gap_change - (winning_party_change - losing_party_change)) > 0.02;

**13b. Coverage at a glance, and the White House party is constant within every CES/ANES wave** (the `MAX()` in
`fct_party_ratings` relies on it). `n_white_house_parties` should be 1 in every row.

In [0]:
%sql
SELECT source, survey_year, COUNT(DISTINCT white_house_party) AS n_white_house_parties, MAX(white_house_party) AS white_house_party,
       COUNT(*) AS n_respondents
FROM int_respondents
GROUP BY source, survey_year
ORDER BY source, survey_year;

In [0]:
%sql
SELECT 'fct_party_ratings'                       AS mart, COUNT(*) AS n_rows FROM fct_party_ratings
UNION ALL SELECT 'fct_wave_contrasts',                     COUNT(*) FROM fct_wave_contrasts
UNION ALL SELECT 'fct_sentiment_monthly',                  COUNT(*) FROM fct_sentiment_monthly
UNION ALL SELECT 'fct_event_study',                        COUNT(*) FROM fct_event_study
UNION ALL SELECT 'fct_swing_timing',                       COUNT(*) FROM fct_swing_timing
UNION ALL SELECT 'fct_fundamentals_at_transitions',        COUNT(*) FROM fct_fundamentals_at_transitions
UNION ALL SELECT 'fct_midterm_placebo',                    COUNT(*) FROM fct_midterm_placebo
UNION ALL SELECT 'fct_pocketbook_gap',                     COUNT(*) FROM fct_pocketbook_gap
UNION ALL SELECT 'fct_expected_vs_actual_inflation',       COUNT(*) FROM fct_expected_vs_actual_inflation
UNION ALL SELECT 'fct_expected_vs_actual_inflation_by_year', COUNT(*) FROM fct_expected_vs_actual_inflation_by_year
UNION ALL SELECT 'fct_rq2_annual_panel',                   COUNT(*) FROM fct_rq2_annual_panel
UNION ALL SELECT 'fct_say_vs_do_monthly',                  COUNT(*) FROM fct_say_vs_do_monthly;